# MyKLPT2

In [1]:
def two_squares_none(n):   #CF source two_squares
    """
    Write the integer `n` as a sum of two integer squares if possible;
    otherwise raise a :exc:`ValueError`.

    INPUT:

    - ``n`` -- integer

    OUTPUT: a tuple `(a,b)` of nonnegative integers such that
    `n = a^2 + b^2` with `a <= b`.

    EXAMPLES::

        sage: two_squares(389)
        (10, 17)
        sage: two_squares(21)
        Traceback (most recent call last):
        ...
        ValueError: 21 is not a sum of 2 squares
        sage: two_squares(21^2)
        (0, 21)
        sage: a, b = two_squares(100000000000000000129); a, b                           # needs sage.libs.pari
        (4418521500, 8970878873)
        sage: a^2 + b^2                                                                 # needs sage.libs.pari
        100000000000000000129
        sage: two_squares(2^222 + 1)                                                    # needs sage.libs.pari
        (253801659504708621991421712450521, 2583712713213354898490304645018692)
        sage: two_squares(0)
        (0, 0)
        sage: two_squares(-1)
        Traceback (most recent call last):
        ...
        ValueError: -1 is not a sum of 2 squares

    TESTS::

        sage: for _ in range(100):                                                      # needs sage.libs.pari
        ....:     a = ZZ.random_element(2**16, 2**20)
        ....:     b = ZZ.random_element(2**16, 2**20)
        ....:     n = a**2 + b**2
        ....:     aa, bb = two_squares(n)
        ....:     assert aa**2 + bb**2 == n

    Tests with numpy and gmpy2 numbers::

        sage: from numpy import int16                                                   # needs numpy
        sage: two_squares(int16(389))                                                   # needs numpy
        (10, 17)
        sage: from gmpy2 import mpz
        sage: two_squares(mpz(389))
        (10, 17)

    ALGORITHM:

    See https://schorn.ch/lagrange.html
    """
    n = ZZ(n)

    if n <= 0:
        if n == 0:
            z = ZZ.zero()
            return (z, z)
        return 

    if n.nbits() <= 32:
        from sage.rings import sum_of_squares
        return sum_of_squares.two_squares_pyx(n)

    # Start by factoring n (which seems to be unavoidable)
    F = n.factor(proof=False)

    # First check whether it is possible to write n as a sum of two
    # squares: all prime powers p^e must have p = 2 or p = 1 mod 4
    # or e even.
    for p, e in F:
        if e % 2 and p % 4 == 3:
            return 

    # We run over all factors of n, write each factor p^e as
    # a sum of 2 squares and accumulate the product
    # (using multiplication in Z[I]) in a^2 + b^2.
    from sage.rings.finite_rings.integer_mod import Mod
    a = ZZ.one()
    b = ZZ.zero()
    for p, e in F:
        if e >= 2:
            m = p ** (e // 2)
            a *= m
            b *= m
        if e % 2:
            if p == 2:
                # (a + bi) *= (1 + I)
                a, b = a - b, a + b
            else:  # p = 1 mod 4
                # Find a square root of -1 mod p.
                # If y is a non-square, then y^((p-1)/4) is a square root of -1.
                y = Mod(2, p)
                while True:
                    s = y**((p - 1) / 4)
                    if not s * s + 1:
                        s = s.lift()
                        break
                    y += 1
                # Apply Cornacchia's algorithm to write p as r^2 + s^2.
                r = p
                while s * s > p:
                    r, s = s, r % s
                r %= s

                # Multiply (a + bI) by (r + sI)
                a, b = a * r - b * s, b * r + a * s

    a = a.abs()
    b = b.abs()
    assert a * a + b * b == n
    return (a, b) if a <= b else (b, a)



def trysolve(f, y):
    y = ZZ(y)
    if y <= 0:
        return
    if y.is_pseudoprime():
        return two_squares_none(y)
    else: 
        return

In [2]:
##def equivalent_prime_ideal(I):
##    Q = I.quaternion_algebra()
##    N0 = I.norm()
##    L = IntegralLattice(I.gram_matrix()).lll().basis_matrix() * I.basis_matrix()
##    bnd = 1
##    while True:
##        for _ in range(5):
##            δ = Q(sum(randrange(-bnd,bnd+1)*v for v in L))
##            N = ZZ(δ.reduced_norm() / N0)
##            if N.is_pseudoprime():
##                break
##        else:
##            bnd += 1
##            continue
##        break
##    print(f'{δ = }')
##    assert δ in I
##    J = I * (δ.conjugate() / N0)
##    del N0
##    print(f'{I = }')
##    print(f'{N = }')
##    return J

In [3]:
def norm_and_generator(I):
    N = ZZ(I.norm())
    O0 = I.left_order()
    bnd = 1
    while True:
        for _ in range(5):
            α = sum(randrange(-bnd,bnd+1)*b for b in I.basis())
            if gcd(α.reduced_norm(), N**2) == N:
                break
        else:
            bnd += 1
            continue
        break
    else:
        assert False
#    print(f'{α = }')
    assert I == O0*N + O0*α
    return N, α

In [4]:
def represent_integer(O0, rhs):
    Q = O0.quaternion_algebra()
    ii,jj,kk = Q.gens()
    if Q.quaternion_order(Q.basis()).discriminant() != 4 * ii**2 * jj**2:
        raise NotImplementedError
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF(1, 0, q)    # x^2 + y^2
    cbnd = isqrt(rhs / 2 / p)
    dbnd = isqrt(rhs / 2 / (p*q))
    if not cbnd or not dbnd:
        print('erreur lN1 trop petit')
        return
    for _ in range(999):
        c = randrange(1,cbnd+1)
        d = randrange(1,dbnd+1)
        rhs1 = rhs - p*nf(c,d)

        sol = trysolve(nf, rhs1)
        if sol is not None:
            a,b = sol
            break
    else:
        print('Pas de solution trouvée')
        return
    γ = Q([a,b,c,d])
#    print(f'{γ = }')
    assert γ in O0
    assert γ.reduced_norm() == rhs
    return γ

In [5]:
def ideal_mod_constraint(N, α, γ):
    ii,jj,kk = α.parent().gens()
    mat = matrix(GF(N), [list(elt) for elt in (γ*jj, γ*kk, α, ii*α, jj*α, kk*α)])
    ker = mat.left_kernel_matrix()
    return next(filter(bool, ker[:,:2].change_ring(ZZ)))  #TODO kernel rank > 1?

In [6]:
def strong_approximation(N, α, C, D, rhs):
    ii,jj,kk = α.parent().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])    # x^2 + y^2
    rhs1 = Mod(rhs, N) / p / nf(C,D)
    if not rhs1.is_square():
        l = next(l for l in rhs.prime_divisors() if not Mod(l,N).is_square())
        rhs //= l
        rhs1 //= l
        assert rhs1.is_square()
    λ = ZZ(rhs1.sqrt())    #lambda n'est pas le bon carré ? approx ?
    print(f'{λ = }')
    λC,λD = (λ * vector(GF(N), (C,D))).change_ring(ZZ)

    x,y,z,t = polygens(ZZ, 'x,y,z,t')
    eqn = rhs - nf(N*x,N*y) - p*nf(λC+N*z, λD+N*t)
    print(f'{eqn =}')
    print(f'{eqn(0,0,z,t) =}')
    assert eqn % N == 0
    eqn1 = eqn // N % N
    U, V, W = eqn1[z], eqn1[t], -eqn1.constant_coefficient()
    assert eqn1 == U*z + V*t - W

    # Petit-Smith
    lat = matrix([[U,0,1,0],[V,0,0,1],[-W,1,0,0]]).stack(N*identity_matrix(4))
    scal = diagonal_matrix([N**2, N, 1, 1])
    for row in matrix(ZZ, filter(bool, (lat * scal).LLL() * ~scal)):
#        print(row)
        if row[1] < 0:
            row = -row
        if not row[0] and row[1] == 1:
            sol0 = row[2:]
            break
    else:
        assert False, 'should never happen'
    import fpylll
    mat = matrix(filter(bool, matrix([[V,-U],[N,0],[0,N]]).LLL()))
    assert mat.dimensions() == (2, 2)
    lat = fpylll.IntegerMatrix(2, 2)
    for i,row in enumerate(mat):
        for j,c in enumerate(row):
            lat[i,j] = c
    gso = fpylll.GSO.Mat(lat)  #TODO lengths are slightly off when q>1
    gso.update_gso()
    cnt = 10
    seen = set()
    count_negatif = 0   #compteur ajouté
    while True:
        enum = fpylll.Enumeration(gso, cnt, fpylll.EvaluatorStrategy.BEST_N_SOLUTIONS)
        rs = enum.enumerate(0, 2, N**2, 0, tuple(mat.solve_left(sol0)))
        for r in rs:
            z,t = sol0 - vector(ZZ,r[1])*mat
            assert eqn1(0,0,z,t) % N == 0
            if (z,t) in seen:
                continue
            seen.add((z,t))
            assert eqn(0,0,z,t) % N**2 == 0
            rhs2 = ZZ(eqn(0,0,z,t)) // N**2
            print(f'{rhs2=}')
            print(rhs2.factor())
            diff = p*nf(λC+N*z, λD+N*t)
            #print(f'{diff =}')
            marge_disc = (diff / p^3).numerical_approx()
            print(f'{marge_disc =}')
            #print(f'{rhs =}')
            #print(f'{λ =}')
            print(f'{z =}', f'{t = }')
            if rhs2 <= 0:
                count_negatif = count_negatif + 1
            else:
                count_negatif = 0
            assert count_negatif < 10
            sol = trysolve(nf, rhs2)          #rhs2 somme de deux carré? positif ? 
            if sol is not None:
                x,y = sol
                break
        else:
            if len(rs) < cnt:
                raise NotImplementedError
            cnt *= 2
            continue
        break

    γ = (λC*jj + λD*kk) + N*(x + y*ii + z*jj + t*kk)
    print(f'{γ = }')
    print(f'{γ.reduced_norm() = }')
    assert γ.reduced_norm() == rhs
    return γ

In [7]:
def liste(facto):
    k = len(facto)
    facto_liste = []
    for i in range(k):
        facto_liste.append([facto[i][0], facto[i][1]])
    return facto_liste

def rand_facto(N,factoN,D,l):

    #On suppose N produit de premier distincts, listés dans factoN.
    #Le discriminant est donné positif
    
    k = len(factoN)
    facto1 = liste(factoN)
    N1 = N
    N2 = 1
        
    liste_indice = [0 .. k-1]
    while N2 <= D^3 and k>0:
        i = liste_indice[randint(0,k-1)]
        prime = facto1[i][0]
        exp = facto1[i][1]
        if exp > 0:
            N2 = N2*prime
            facto1[i][1] = exp-1
            N1 = N1 // prime
        else:
            liste_indice.remove(i)
        k = len(liste_indice)
    
    assert N == N1*N2
    if l*N1 > 3*D and N2 > D^3:   #3 pour avoir une marge d'essais pour Cornacchia
        return N1, N2    #On préfère tj N2 plus grand
    else:
        return rand_facto(N,factoN,D,l)

In [8]:
def klpt(I, N, factoN):
    print(f'{N = }')
    ii,jj,kk = I.quaternion_algebra().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])  #Peut-être jj plutôt ??
    gcdN = N.gcd(p)
    
    FactoG = gcdN.factor(proof=False)
    for p, e in FactoG:
        if e % 2 and p % 4 == 3:
            print(FactoG)
            raise ValueError('Gcd bloquant Cornacchia')
            
    while True:   #Necesaire ?

        if not ZZ(I.norm()).is_pseudoprime():
            raise NotImplementedError         #On suppose I de norme premier, quitte a faire une equivalence. 
#            I = equivalent_prime_ideal(I)

        l,α = norm_and_generator(I)   
        print(l)
        O0 = I.left_order()
        abs_disc = O0.discriminant()   #discriminant positif
            
        test = True
        while test:
            N1, N2 = rand_facto(N,factoN,abs_disc,l)   #TODO : Tester si la facto est déjà vue ?
            assert N1*N2 == N
            print('Tentative repinteger')
            print(f'{N1 = }')
            print(f'{N2 = }')
            γ = represent_integer(O0, l*N1)
            if γ is not None:
                test = False
        print('Marge erreur sur N2', (N2/(abs_disc)^3).numerical_approx())
        print('gcd N2 disc', N2.gcd(abs_disc))
        
        C,D = ideal_mod_constraint(l, α, γ)
        print(f'{C = }')
        print(f'{D = }')
        if l.divides(nf(C,D)):
            raise NotImplementedError('bad')

        µ = strong_approximation(l, α, C, D, N2)
        
        return γ * µ

# Test Torsions

## CONTEXTE : Courbe avec anneaux d'endomorphisme

In [23]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.libs.libecm import ecmfactor
from sage.misc.search import search
import time

In [ ]:
# CONTEXTE 1 :
db = HilbertClassPolynomialDatabase() #discriminant jusque -9999

def curve_from_disc(D,p):
    #on suppose D negatif
    H = db[D]
    Fp = GF(p)
    PFp.<Xp> = PolynomialRing(Fp)
    H = PFp(H)
    Hr = H.factor()[0][0]
    d = Hr.degree()
    Fd = GF(p^d)
    PFd.<Xd> = PolynomialRing(Fd)
    Hr = Hr(Xd)
    Hj = (Hr.factor()[0][0])
    j = -Hj(0)
    E = EllipticCurve_from_j(j)
    return E
    
D = -1111
p = 109
E = curve_from_disc(D,p)
j = E.j_invariant()
H = db[-D]
assert H(j) == 0
Fq = E.base_ring()
dq = Fq.degree()
q = p^dq
P = E.random_point()

f = 3 # obtenue avec -9999 = - 3^2*11*101
K.<rK> = QuadraticField(-1111)
rD = 3*rK
dK = K.discriminant()
wK = (dK + rK)/2
OK = K.maximal_order()
O = K.order([1,f*wK])
assert f == O.conductor()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*q
fm = sqrt(Dm/(K.discriminant()))



In [15]:
# CONTEXTE 2 : Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

#l = 5000000029 exemple de calcul d'isogénie, cf article

In [ ]:
#CONTEXTE 3 : exemples Sutherland

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

#Exemple 2:

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

#CONTEXTE 4 : Sutherland database

#exemple 1

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

#exemple 2

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

## ETUDE DE LA TORSION DE E

In [18]:
def etude_torsions(E,p,O,K,Nk_max,Nk_min):
    
    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    rD = f*rK
    wK = (dK + rK)/2
    CE = E.cardinality_pari()
    tracef = E.trace_of_frobenius()
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    s = int((-fm*dK + tracef)/2)
    s2 = (-fm*dK - tracef)/2
    frob = fm*wK + s
    frob2 = -(fm*wK + s2)
    assert (frob^2 - tracef*frob + p) == 0
    assert (frob2^2 - tracef*frob2 + p) == 0

    if dK%4 == 1 :
        a = int((tracef - fm)/2)
    else :
        a = int(tracef/2)
    Nmax = gcd(a-1,fm/f)

    torsions = []
    if Nmax > 2*Nk_min:             #solution à clapoti n'existe pas si N < N1 + N2
        torsions.append([Nmax,1])

    #traces = [tracef]
    #cards = [CE]
    
    frobd = frob
    frob2d = frob2
    q = p
    d = 1
    
    while Nmax < Nk_max^2:
        
        d = d+1
        q = q*p
        frobd = frobd*frob
        frob2d = frob2d*frob2
        tracefd = frobd + frob2d
        
        Dm = tracefd^2 - 4*q
        fm = int(sqrt(Dm/(dK)))
        assert (frobd^2 - tracefd*frobd + q) == 0
        assert (frob2d^2 - tracefd*frob2d + q) == 0
        
        if dK%4 == 1 :
            a = int((tracefd - fm)/2)
        else :
            a = int(tracefd/2)
            
        Nmax = gcd(a-1,fm/f)
        if Nmax > 2*Nk_min:
            torsions.append([Nmax,d])
        #traces.append(tracefd)
        #cards.append(q + 1 - tracefd)
    
    return torsions, d

In [ ]:
#Contexte choisi : 

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

D = O.discriminant()

In [19]:
torsions, deg_max = etude_torsions(E,p,O,K,isqrt(-D),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(120,
 85,
 [[3, 2],
  [1042, 3],
  [3, 4],
  [18756, 6],
  [3, 8],
  [1042, 9],
  [3, 10],
  [3013, 11],
  [37512, 12],
  [129, 14],
  [1042, 15],
  [3, 16],
  [1069092, 18],
  [15, 20],
  [1042, 21],
  [9039, 22],
  [75024, 24],
  [3, 26],
  [1042, 27],
  [129, 28],
  [581436, 30],
  [3, 32],
  [3139546, 33],
  [3, 34],
  [71, 35],
  [2138184, 36],
  [3, 38],
  [163594, 39],
  [15, 40],
  [5645556, 42],
  [804471, 44],
  [1042, 45],
  [3, 46],
  [150048, 48],
  [3, 50],
  [1042, 51],
  [3, 52],
  [3207276, 54],
  [3013, 55],
  [129, 56],
  [238618, 57],
  [3, 58],
  [354675960, 60],
  [3, 62],
  [1042, 63],
  [3, 64],
  [3786292476, 66],
  [269, 67],
  [3, 68],
  [1042, 69],
  [9159, 70],
  [72698256, 72],
  [3, 74],
  [1042, 75],
  [3, 76],
  [3013, 77],
  [38280996, 78],
  [15, 80],
  [1042, 81],
  [3, 82],
  [11291112, 84],
  [3, 86],
  [1042, 87],
  [804471, 88],
  [33141852, 90],
  [3, 92],
  [1357726, 93],
  [849, 94],
  [300096, 96],
  [129, 98],
  [3139546, 99],
  [75, 100],


# TEST SOLUTIONS KLPT

In [27]:
def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"





def sol_KLPT(N,factoN,aa):

    #Utiliser KLPT pour résoudre l'equation définie par N et aa
    while True :
    
        l, α = aa.gens_two()
        dK = α.parent().number_field().discriminant()
        Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
        assert (α[0] + α[1]*j).reduced_norm() == α.norm()
        assert (α[0] + α[1]*j).reduced_trace() == α.trace()
        assert j.reduced_norm() == rK.norm() and j.reduced_trace() == rK.trace()
        r = α.parent().number_field().gen()
        assert (r+1)/2 in O   #Necessaire ? ordre de discriminant = 1 mod 4 ?
        OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
        I = OO*l + OO*(α[0] + α[1]*j)
        print(f'{I = }')


        elt = klpt(I,N,factoN)
        assert elt in I
        print(f'{elt = }', '| norm:', elt.reduced_norm().factor())
    

        b = elt[0] + elt[2]*r
        c = elt[1] + elt[3]*r
        #while b and c and b/2 in aa and c/2 in aa:  # can we avoid this a priori in KLPT?
            #b /= 2
            #c /= 2
        print(f'{b = }')
        print(f'{c = }')
        assert b in aa
        assert c in aa
        bb = O.ideal([g*b.conjugate()/aa.norm() for g in aa.gens()])
        cc = O.ideal([g*c.conjugate()/aa.norm() for g in aa.gens()])
        Nb = bb.norm()
        Nc = cc.norm()
        print(f'{Nb = }')#.factor())
        print(f'{Nc = }')#.factor())
        if ZZ(Nb + Nc) == N:
            break
        else:
            print('Erreur sur Nb, Nc')
    if Nb.gcd(Nc) == 1:
        print('solution premier entre eux')
        return Nb, Nc, 1
    else :
        print('solution avec gcd>1')
        return Nb, Nc, Nb.gcd(Nc)
        
        

In [28]:
def reduction_ideal(aa):
    qa = aa.quadratic_form()
    ra = qa.reduced_form()
    lr = ra.small_prime_value()
    aar = ideal_de_norme(lr,f,D)
    if aar.is_equivalent(aa):
        return aar
    else: 
        return aar.conjugate()


In [29]:
l =randint(10^20, 10^21)
aa = ideal_de_norme(l,f,D)
aa = reduction_ideal(aa)
aa.norm()

7

In [31]:
k_candidats = len(torsions)
for i in [ 0 .. k_candidats - 1]:
    N = torsions[i][0]
    factoN = torsions[i][1]
    Nb, Nc, verif = sol_KLPT(N,factoN,aa)
    

I = Fractional ideal (7/2 + 1/2*j, 7/2*i + 1/2*k, j, k)
N = 3


TypeError: factor() got an unexpected keyword argument 'proof'

In [33]:
aa = ideal_de_norme(l,f,D)
aa = reduction_ideal(aa)
dK = α.parent().number_field().discriminant()
l, α = aa.gens_two()
Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
I = OO*l + OO*(α[0] + α[1]*j)
Q = OO.quaternion_algebra()
ii,jj,kk = Q.gens()
if Q.quaternion_order(Q.basis()).discriminant() != 4 * ii**2 * jj**2:
    raise NotImplementedError

q, p = ZZ(-ii**2), ZZ(-jj**2)

cbnd = isqrt(l*2 / 2 / p)
dbnd = isqrt(l*2 / 2 / (p*q))
cbnd, dbnd, p, q

(0, 0, 38669866235, 1)

In [41]:
int(log(sqrt(p^7),2))

123

In [77]:
N = 2^130 + randint(1,2^10)   #Taille de N pour avoir un rhs2 positif : log(D^(3,5),2) = 52
# Exemple qui fonctionne 2^180 :N = 1532495540865888858358347027150309183618739122183602389
# Exemple qui fonctionne 2^130 :N = 1361129467683753853853498429727072846691
factoN = N.factor()
print(factoN)
print(p.factor())
sol_KLPT(N,factoN,aa)

31547 * 419999 * 1514197 * 67843888662238392103307
10000000019
I = Fractional ideal (1/2 + 1066081/2*j, 1/2*i + 1066081/2*k, 1080229*j, 1080229*k)
N = 1361129467683753853853498429727072845987
1080229
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 

KeyboardInterrupt: 

In [56]:
N.gcd(p)

77

# TEST SOLUTION PEGASIS

## Brouillon

In [9]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal

#Contexte choisi : 

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()


rK = K.gens()[0]
dK = K.discriminant()
f = O.conductor()
D = (f^2)*dK
rD = f*rK
wK = (dK + rK)/2
CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))


In [10]:
def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

In [11]:
l = next_prime(randint(10^20, 10^21))
while kronecker(D,l) != 1:
        l = next_prime(l)
    
L = ideal_de_norme(l,f,D)
L

Ideal (12529326030653621355/2*rK + 1/2, 392741390943279979141*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 10000006055889179 with rK = 1.000000302794413?e8*I

In [12]:
#TODO :

#def calcul de polynôme modulaire, ou accès à database Sutherland ?
#def is_Bgood #Pour tester Pegasis4D



In [13]:
def ideal_to_element(a,L,O):
    #On suppose a equivalent à L. On cherche alpha dans L tel que a = (alphabar / N(L))L
    assert a.is_equivalent(L)
    B = a*(L.conjugate())
    alphabar = (B.gens_reduced())[0]
    alpha = alphabar.conjugate()
    assert NumberFieldOrderIdeal(O,alphabar) == B
    assert alpha in L
    return alpha

def element_to_ideal(alpha,L,O):
    #On suppose alpha dans L et on lui asocie un idéal equivalent
    assert alpha in L
    alphabar = alpha.conjugate()
    B = NumberFieldOrderIdeal(O,alphabar)
    a2 = B*L
    g1, g2 = a2.gens_reduced()
    a = NumberFieldOrderIdeal(O,[g1/(L.norm()), g2/(L.norm())])
    assert a.is_equivalent(L)
    return a

qL = L.quadratic_form()
rL = qL.reduced_form()
a = NumberFieldOrderIdeal(O,[rL[0], (-rL[1] + f*rK)/2])
rL, a.quadratic_form()   #TODO WTF pas la même pour norme 23 ?? Probleme important ? a.quadratic_form() n'est pas réduite

(49013217*x^2 + 19302415*x*y + 52907103*y^2,
 49013217*x^2 - 78724019*x*y + 82617905*y^2)

In [14]:
a == NumberFieldOrderIdeal(O,a.quadratic_form())

True

In [15]:
alpha = ideal_to_element(a,L,O)
a2 = element_to_ideal(alpha,L,O)
a == a2

True

In [16]:
def base_vecteurs_courts(L,O):
    f = O.conductor()
    qL = L.quadratic_form()
    rL = qL.reduced_form()
    a = NumberFieldOrderIdeal(O,[rL[0], (-rL[1] + f*rK)/2])
    b = NumberFieldOrderIdeal(O,[rL[2], (rL[1] + f*rK)/2])
    alpha = ideal_to_element(a,L,O)
    beta = ideal_to_element(b,L,O)
    assert NumberFieldOrderIdeal(O,[alpha,beta]) == L  #TODO possiblement faux ?
    return alpha, beta

base_vecteurs_courts(L,O)

(-776602*rK + 114971338614341, 2651563/2*rK + 113169478591379/2)

In [17]:
#Remarque : on peut réduire L avant de calculer la base de vecteurs courts, i.e qL = rL et a = L

base_vecteurs_courts(a,O)

(49013217, 1/2*rK - 19302415/2)

In [25]:
def liste_premiers_splits(D,borne_B,fm,p):
    #Borne_B: taille maximale des nombres premiers regardés
    liste_B = []
    i = 2
    while i < borne_B :
        if kronecker(D,i) == 1 and i.gcd(fm*p) == 1:
            liste_B.append(i)
        i = next_prime(i)
    return liste_B
    
prime_max = 370 #Nb de pol modulaire dans la base de donnée de sagemath (Kohel, PARI/GP)

liste_B = liste_premiers_splits(D,prime_max,fm,p)  #assurer une taille minimale de la liste ? 
len(liste_B)

23

In [58]:
def make_liste_ideq(L,O,m,liste_B,CE):
    # m nombre d'idéaux que l'on teste (2m^2)
    # présence de CE : pour evaluer des endomorphismes, on veut des normes premieres avec le cardinal
    # On suppose que l'on ne veut evaluer l'isogenie que sur des points du corps de base Fq.
    alpha, beta = base_vecteurs_courts(L,O)
    liste_ideq = []
    Nk_max = 1
    Nk_min = -(O.discriminant())
    for x in [0 .. m]:   #on evite de creer gamma et -gamma
        for y in [-m .. m]:
            if (x != 0 or y > 0):   #on evite les cas x = 0 et y <= 0
                gamma = x*alpha + y*beta
                if gamma == 0:
                    raise ValueError('erreur base courte liée')
                I = element_to_ideal(gamma,L,O)
                NI = I.norm()
                if NI.gcd(CE) != 1:
                    continue
                Nk = NI
                Ne = 1
                Ne_facto = []
                for p in liste_B:
                    exp = 0
                    while Nk%p == 0:
                        exp = exp + 1
                        Nk = Nk/p
                        Ne = Ne*p
                    if exp > 0:
                        Ne_facto.append([p,exp])
                assert NI == Nk*Ne
                if Nk == 1:
                    print('Ideal equivalent friable', I)
                liste_ideq.append([I,Nk,Ne,Ne_facto])
                if Nk > Nk_max:
                    Nk_max = Nk
                if Nk < Nk_min:
                    Nk_min = Nk
    return liste_ideq, Nk_max, Nk_min




In [59]:
m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

liste_ideq, Nk_max, Nk_min = make_liste_ideq(a,O,m_id,liste_B,CE)
#[b_prep[2] for b_prep in liste_ideq] afficher les Ne

In [60]:
len(liste_ideq), (Nk_max/(isqrt(-D))).numerical_approx(), Nk_min

(544, 292.047630315711, 27)

In [77]:
def coin_equation(N1,N2,N):
    #On veut résoudre uN1 + vN2 == N dans NN, avec uN1 gcd vN2 == 1 et N divise Nmax
    #assert N >= N1*N2  necessaire ?
    d,u0,v0 = xgcd(N1,N2)
    assert d == 1
    
    if u0 > 0:
        u, v, N, sol = coin_equation(N2,N1,N)
        return v, u, N, sol

    #On suppose u0 <= 0
        
    ku = int(((-u0)*N) // N2) + 1  #erreur de int, etrange ?
    kv = int((v0*N)//N1)
    u = u0*N + ku*N2
    v = v0*N - ku*N1
    assert u*N1 + v*N2 == N

    sol = False
    if ku > kv :
        #print('echec equation dans NN')
        return u, v, N, sol
        
    nb_sols = kv - ku + 1

    while sol == False and nb_sols >= 0:
        duv = u.gcd(v)
        if N%duv != 0:
            continue
        if ((u/duv)*N1).gcd((v/duv)*N2) == 1:
            u = u/duv
            v = v/duv
            N = N/duv
            break
        u = u + N2
        v = v - N1
        nb_sols = nb_sols - 1

    assert u*N1 + v*N2 == N
    if (u*N1).gcd(v*N2) == 1 and v > 0 and u > 0:
        sol = True
            
    return u, v, N, sol


def clapoti_equation(torsions,liste_ideq):
    k_id = len(liste_ideq)
    sols = []
    sols_ext = []
    for i in [0 .. k_id-1]:
        b_prep = liste_ideq[i]
        b, N1, M1, factoM1 = b_prep
        for j in [i .. k_id-1]:
            c_prep = liste_ideq[j]
            c, N2, M2, factoM2 = c_prep
            if N1.gcd(N2) != 1:
                #print(N1,N2,'echec N1 N2 pas premier entre eux')
                continue

            for Nmax, ext in torsions :
                sol_ext = False
                #print('Nmax =', Nmax)
                
                Gcd1 = Nmax.gcd(M1)
                Nmax_loc = Nmax/Gcd1
                Gcd2 = Nmax_loc.gcd(M2)
                Nmax_loc = Nmax_loc/Gcd2
                #print('Nmax_loc =', Nmax_loc)
                
                Nmin = N1+N2   #N1*N2 necessaire ?
                if Nmin > Nmax_loc :
                    #print(Nmin, 'echec Nmin')
                    continue
                
                #print('Tentative N =',Nmax_loc, 'N1 =', N1, 'N2 =',N2)
                u,v,N,sol = coin_equation(N1,N2,Nmax_loc)
                if sol:
                    #print('solution :',u,'*',N1,'+',v,'*',N2,'=',N)
                    sols.append([u,N1,v,N2,N,i,j])
                    sol_ext = True
                #else:
                    #print(N,N1,N2,'echec equation')
                if sol_ext == True and ext not in sols_ext:
                    sols_ext.append(ext)

    return sols, sols_ext

In [29]:
coin_equation(2,3,31)

(2, 9, 31, True)

In [ ]:
sols, sols_ext = clapoti_equation(torsions,liste_ideq)

In [219]:
len(sols)

140

In [220]:
sols_ext.sort()
sols_ext

[84, 96, 108, 112, 116, 120]

In [ ]:
sols

In [222]:
liste_ideq[0]

[Ideal (2881277590231/2*rK + 1/2, 1817896251209*rK) of Order of conductor 852857 generated by 852857/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 7 with rK = 2.645751311064591?*I,
 2131537,
 1,
 []]

## Bilan

In [8]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
import time

def etude_torsions(E,p,O,K,Nk_max,Nk_min):
    
    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    rD = f*rK
    wK = (dK + rK)/2
    CE = E.cardinality_pari()
    tracef = E.trace_of_frobenius()
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    s = int((-fm*dK + tracef)/2)
    s2 = (-fm*dK - tracef)/2
    frob = fm*wK + s
    frob2 = -(fm*wK + s2)
    assert (frob^2 - tracef*frob + p) == 0
    assert (frob2^2 - tracef*frob2 + p) == 0

    if dK%4 == 1 :
        a = int((tracef - fm)/2)
    else :
        a = int(tracef/2)
    Nmax = gcd(a-1,fm/f)

    torsions = []
    if Nmax > 2*Nk_min:             #solution à clapoti n'existe pas si N < N1 + N2
        torsions.append([Nmax,1])

    #traces = [tracef]
    #cards = [CE]
    
    frobd = frob
    frob2d = frob2
    q = p
    d = 1
    
    while Nmax < Nk_max^2:
        
        d = d+1
        q = q*p
        frobd = frobd*frob
        frob2d = frob2d*frob2
        tracefd = frobd + frob2d
        
        Dm = tracefd^2 - 4*q
        fm = int(sqrt(Dm/(dK)))
        assert (frobd^2 - tracefd*frobd + q) == 0
        assert (frob2d^2 - tracefd*frob2d + q) == 0
        
        if dK%4 == 1 :
            a = int((tracefd - fm)/2)
        else :
            a = int(tracefd/2)
            
        Nmax = gcd(a-1,fm/f)
        if Nmax > 2*Nk_min:
            torsions.append([Nmax,d])
        #traces.append(tracefd)
        #cards.append(q + 1 - tracefd)
    
    return torsions, d

def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

def ideal_to_element(a,L,O):
    #On suppose a equivalent à L. On cherche alpha dans L tel que a = (alphabar / N(L))L
    assert a.is_equivalent(L)
    B = a*(L.conjugate())
    alphabar = (B.gens_reduced())[0]
    alpha = alphabar.conjugate()
    assert NumberFieldOrderIdeal(O,alphabar) == B
    assert alpha in L
    return alpha

def element_to_ideal(alpha,L,O):
    #On suppose alpha dans L et on lui asocie un idéal equivalent
    assert alpha in L
    alphabar = alpha.conjugate()
    B = NumberFieldOrderIdeal(O,alphabar)
    a2 = B*L
    g1, g2 = a2.gens_reduced()
    a = NumberFieldOrderIdeal(O,[g1/(L.norm()), g2/(L.norm())])
    assert a.is_equivalent(L)
    return a

def base_vecteurs_courts(L,O):    #N'utilise pas LLL. Mieux ?
    f = O.conductor()
    qL = L.quadratic_form()
    rL = qL.reduced_form()
    a = NumberFieldOrderIdeal(O,[rL[0], (-rL[1] + f*rK)/2])
    b = NumberFieldOrderIdeal(O,[rL[2], (rL[1] + f*rK)/2])
    alpha = ideal_to_element(a,L,O)
    beta = ideal_to_element(b,L,O)
    assert NumberFieldOrderIdeal(O,[alpha,beta]) == L  #alpha, beta de normes premier entre elles, donc libre
    return alpha, beta

def liste_premiers_splits(D,borne_B,fm,p):
    #Borne_B: taille maximale des nombres premiers regardés
    liste_B = []
    i = 2
    while i < borne_B :
        if kronecker(D,i) == 1 and i.gcd(fm*p) == 1:
            liste_B.append(i)
        i = next_prime(i)
    return liste_B

def make_liste_ideq(L,O,m,liste_B,CE):
    # m nombre d'idéaux que l'on teste (2m^2)
    # présence de CE : pour evaluer des endomorphismes, on veut des normes premieres avec le cardinal
    # On suppose que l'on ne veut evaluer l'isogenie que sur des points du corps de base Fq.
    
    alpha, beta = base_vecteurs_courts(L,O)
    liste_ideq = []
    Nk_max = 1
    Nk_min = -(O.discriminant())
    ideal_friable = False
    
    for x in [0 .. m]:   #on evite de creer gamma et -gamma
        for y in [-m .. m]:
            if (x != 0 or y > 0):   #on evite les cas x = 0 et y <= 0
                gamma = x*alpha + y*beta
                
                if gamma == 0:
                    raise ValueError('erreur base courte liée') 
                    
                I = element_to_ideal(gamma,L,O)
                NI = I.norm()
                if NI.gcd(CE) != 1:
                    continue
                    
                Nk = NI
                Ne = 1
                Ne_facto = []
                for p in liste_B:
                    exp = 0
                    while Nk%p == 0:
                        exp = exp + 1
                        Nk = Nk/p
                        Ne = Ne*p
                    if exp > 0:
                        Ne_facto.append([p,exp])
                        
                assert NI == Nk*Ne
                if Nk == 1:
                    #print('Ideal equivalent friable', I)
                    ideal_friable = True
                liste_ideq.append([I,Nk,Ne,Ne_facto])
                if Nk > Nk_max:
                    Nk_max = Nk
                if Nk < Nk_min:
                    Nk_min = Nk
                    
    return liste_ideq, Nk_max, Nk_min, ideal_friable

def coin_equation(N1,N2,N):
    
    #On veut résoudre uN1 + vN2 == N dans NN, avec uN1 gcd vN2 == 1 et N divise Nmax
    
    d,u0,v0 = xgcd(N1,N2)
    assert d == 1
    
    if u0 > 0:
        u, v, N, sol = coin_equation(N2,N1,N)
        return v, u, N, sol

    #On suppose u0 <= 0
        
    ku = int(((-u0)*N) // N2) + 1  #erreur de int, etrange ?
    kv = int((v0*N)//N1)
    u = u0*N + ku*N2
    v = v0*N - ku*N1
    assert u*N1 + v*N2 == N

    sol = False
    if ku > kv :
        #print('echec equation dans NN')
        return u, v, N, sol
        
    nb_sols = kv - ku + 1

    while sol == False and nb_sols >= 0:
        duv = u.gcd(v)
        if N%duv != 0:
            nb_sols = nb_sols - 1
            continue
        if ((u/duv)*N1).gcd((v/duv)*N2) == 1:
            u = u/duv
            v = v/duv
            N = N/duv
            break
        u = u + N2
        v = v - N1
        nb_sols = nb_sols - 1

    assert u*N1 + v*N2 == N
    if (u*N1).gcd(v*N2) == 1 and v > 0 and u > 0:
        sol = True
            
    return u, v, N, sol


def clapoti_equation(torsions,liste_ideq):
    k_id = len(liste_ideq)
    sols = []
    sols_ext = []
    for i in [0 .. k_id-1]:
        b_prep = liste_ideq[i]
        b, N1, M1, factoM1 = b_prep
        
        for j in [i .. k_id-1]:
            c_prep = liste_ideq[j]
            c, N2, M2, factoM2 = c_prep
            
            if N1.gcd(N2) != 1:
                #print(N1,N2,'echec N1 N2 pas premier entre eux')
                continue

            for Nmax, ext in torsions :
                sol_ext = False
                
                Gcd1 = Nmax.gcd(M1)
                Nmax_loc = Nmax/Gcd1
                Gcd2 = Nmax_loc.gcd(M2)
                Nmax_loc = Nmax_loc/Gcd2
                
                Nmin = N1+N2   
                if Nmin > Nmax_loc :
                    continue
                
                #print('Tentative N =',Nmax_loc, 'N1 =', N1, 'N2 =',N2)
                u,v,N,sol = coin_equation(N1,N2,Nmax_loc)
                if sol:
                    #print('solution :',u,'*',N1,'+',v,'*',N2,'=',N)
                    sols.append([u,N1,v,N2,N,i,j])
                    sol_ext = True
                #else:
                    #print(N,N1,N2,'echec equation')
                if sol_ext == True and ext not in sols_ext:
                    sols_ext.append(ext)

    return sols, sols_ext

In [9]:
def first_solutions_clapoti(prime_max,m_id,l,E,K,O):
    #deg-max : degré d'extension maximal regardé
    #prime_max : taille maximal des nombres premiers considérés petit
    #m_id : interval sur lequel on cherche des idéaux équivalent

    #Renvoie la premiere solution trouvée (plus petite extension possible)

    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    CE = E.cardinality_pari()
    p = E.base_ring().characteristic()
    tracef = p + 1 - CE   #E définie sur Fp
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    
    L = ideal_de_norme(l,f,D)
    
    resultat = []
    liste_B = liste_premiers_splits(D,prime_max,fm,p)
    liste_ideq, Nk_max, Nk_min, ideal_friable = make_liste_ideq(L,O,m_id,liste_B,CE)
    torsions_candidats, deg_max = etude_torsions(E,p,O,K,Nk_max,Nk_min)
    
    for torsion in torsions_candidats:
        sols, sols_ext = clapoti_equation([torsion],liste_ideq)
        if len(sols)>0:
            resultat.append(torsion)
            resultat.append(sols)
            break
    return resultat, liste_ideq, deg_max, ideal_friable

def test_solutions_clapoti(prime_max,m_id,l,E,K,O):
    #prime_max : taille maximal des nombres premiers considérés petit
    #m_id : interval sur lequel on cherche des idéaux équivalent

    #Mesure l'efficacité d'augmenter prime_max
    
    resultat = []
    Bp = 1
    while Bp <= prime_max:
        first, liste_ideq, deg_max, ideal_friable = first_solutions_clapoti(Bp,m_id,l,E,K,O)
        resultat.append([Bp,deg_max,len(liste_ideq),first[0],len(first[1])])
        Bp = next_prime(Bp)
    return resultat

def stat_solutions_clapoti(prime_max, m_id, nb_essais,E,K,O): 

    print('prime_max :', prime_max)

    moy_deg = 0
    moy_id = 0
    nb_id_friable = 0
    max_deg = 0
    min_deg = 0
    D = O.discriminant()

    print('Nombre essais', nb_essais)

    non_friable = 0

    while non_friable < 20 and nb_id_friable < 200:

        l = next_prime(randint(10^20, 10^21))
        while kronecker(D,l) != 1:
            l = next_prime(l)
        first, liste_ideq, deg_max, ideal_friable = first_solutions_clapoti(prime_max,m_id,l,E,K,O)
        deg_sol = first[0][1]
        moy_id = moy_id + len(liste_ideq)

        if ideal_friable:
            print('friable', nb_id_friable)
            nb_id_friable += 1
        else:
            non_friable += 1
            print('non_friable', non_friable)
            moy_deg = moy_deg + deg_sol
            if deg_sol > max_deg:
                max_deg = deg_sol
            if deg_sol < min_deg or min_deg == 0:
                min_deg = deg_sol

    print('degrés moyen de first solution :', moy_deg/20)
    print('dégrés maximal parmis les solutions :', max_deg)
    print('degrés minimal parmis les solutions :', min_deg)
    print('Nb idéaux théoriques :', m_id*m_id*2 + m_id)
    print('Nb idéaux retenus en moyenne :', moy_id/(nb_id_friable + 20))
    print('Nb de tentatives ignorées (idéal friable) :', nb_id_friable)
    print('Fréquence idéaux friables :', nb_id_friable/(nb_id_friable + 20) )
    print('\n')

    return 



In [6]:
#Exemple de Jao "small"

print('Exemple JaoSmall')

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 

#stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max = 1000           # taille max polynome modulaire Sutherland  

#stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

Exemple JaoSmall
Discriminant = -38669866235


In [ ]:
#EXEMPLES Bisson Sutherland

print('Exemple BS1')

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)

D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 

#stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

prime_max = 1000           # taille max polynome modulaire Sutherland  

#stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)






#Exemple 2:

print('Exemple BS2')

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 

#stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

In [5]:
#EXEMPLES Sutherland database

#exemple 1

print('Exemple Sutherland1')

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)





#exemple 2

print('Exemple Sutherland2')

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

Exemple Sutherland1
Discriminant = -102197306669747
prime_max : 370
Nombre essais 20
non_friable 1
non_friable 2
non_friable 3
friable 0
non_friable 4
non_friable 5
non_friable 6
friable 1
non_friable 7
friable 2
friable 3
non_friable 8
non_friable 9
non_friable 10
non_friable 11
friable 4
non_friable 12
non_friable 13
non_friable 14
non_friable 15
non_friable 16
non_friable 17
non_friable 18
non_friable 19
non_friable 20
degrés moyen de first solution : 6
dégrés maximal parmis les solutions : 6
degrés minimal parmis les solutions : 6
Nb idéaux théoriques : 406
Nb idéaux retenus en moyenne : 420
Nb de tentatives ignorées (idéal friable) : 5
Fréquence idéaux friables : 1/5


prime_max : 1000
Nombre essais 20
friable 0
non_friable 1
friable 1
friable 2
friable 3
friable 4
friable 5
friable 6
friable 7
friable 8
friable 9
friable 10
friable 11
friable 12
friable 13
friable 14
non_friable 2
friable 15
friable 16
friable 17
friable 18
non_friable 3
friable 19
friable 20
friable 21
non_friab

In [ ]:
# CONTEXTE Possible : Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

#l = 5000000029 exemple de calcul d'isogénie, cf article


#EXEMPLES Bisson Sutherland

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

#Exemple 2:

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

#EXEMPLES Sutherland database

#exemple 1

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

#exemple 2

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

In [29]:
#Contexte choisi :

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

D = O.discriminant()

In [30]:
l = next_prime(randint(10^20, 10^21))
while kronecker(D,l) != 1:
        l = next_prime(l)
    
m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis
prime_max = 370           # taille max polynome modulaire sagemath. Acces Sutherland possible ? 
#prime_max = 2*int(log(-(O.discriminant()),2))^2  #choix de la borne théorique utilisée dans Broker/Jao ?

In [31]:
2*int(log(-(O.discriminant()),2))^2   #prime_max au sens de l'algo Broker Charles Lauter

5618

In [32]:
first, liste_ideq, deg_max = first_solutions_clapoti(prime_max,m_id,l,E,K,O)

In [33]:
first

[[20971440, 72], [[127, 859, 9, 5139, 155344, 138, 443]]]

In [34]:
test_prime_max = test_solutions_clapoti(prime_max,m_id,l,E,K,O)

In [35]:
test_prime_max

[[1,
  1092,
  544,
  [[112117908636558720, 576],
   [[1941343, 19962227561, 4605647, 15929222951, 112117908636558720, 16, 439],
    [1592761, 15278487283, 307440547, 103187351, 56058954318279360, 18, 31],
    [10867, 9427416241, 272641, 1256100893, 444912335859360, 21, 28],
    [4072433, 4985792449, 889775753, 103187351, 112117908636558720, 24, 31],
    [9919457, 4985792449, 56142379, 117604573, 56058954318279360, 24, 33],
    [7572179, 4985792449, 3798673, 4818956893, 56058954318279360, 24, 358],
    [3162293, 1953615907, 60856679, 103187351, 12457545404062080, 27, 31],
    [6461279, 1953615907, 210449119, 117604573, 37372636212186240, 27, 33],
    [19936481, 1953615907, 380579203, 192258751, 112117908636558720, 27, 64],
    [4598869, 1953615907, 5810249, 3277837753, 28029477159139680, 27, 225],
    [1951819, 1256100893, 338422823, 103187351, 37372636212186240, 28, 31],
    [1808729, 1256100893, 86608871, 117604573, 12457545404062080, 28, 33],
    [69299, 1256100893, 10547, 178486345

# Ideaux friables et isogénies

In [41]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.schemes.elliptic_curves.hom_velusqrt import EllipticCurveHom_velusqrt
from sage.schemes.elliptic_curves.weierstrass_morphism import *
from sage.groups.generic import order_from_multiple
from sage.schemes.elliptic_curves.ell_curve_isogeny import compute_isogeny_bmss
from sage.schemes.elliptic_curves.hom_frobenius import EllipticCurveHom_frobenius
from sage.libs.libecm import ecmfactor
from sage.misc.search import search
import time

def forme_de_norme(l,D):

    #Remarque : Seysen dmande une racine spécifique, la plus petite vu dans N. Mais cela n'est pas nécessaire pour la suite (Cf Cohen)
    
    if kronecker(D,l) == 1:
        if l == 2 :
            d = Mod(D,8)
            b = square_root_mod_prime_power(d,2,3)
            b = ZZ(b)
        else :
            x = Mod(D,4)
            d = Mod(D,l)
            y = square_root_mod_prime(d,l)
            b = x.crt(y)
            b = ZZ(b)
        ql = BinaryQF([l,b, int(int((b^2 - D))/int(4*l))])
        return ql
    else:
        raise ValueError('l pas split')

def base_fwk(b,dk,f):
    #Dans K = Q(rk) quadratique de discriminant dk, on se donne b = x + y*rk dans un ordre O
    #On veut écrire b dans la base [1,f*wk] de l'ordre O, wk = (dk + rk)/2
    #xw, yw sont dans ZZ
    
    x = b[0]
    y = b[1]
    xw = x - dk*y
    yw = (2*y)/f
    return xw,yw
    
def base_fq1(b,fm,dk,f,tracef):
    
    # On représente le froebenius fq par le complexe (tracef + fm*t)/2 dans K = Q(t).
    # Remarque : il est a priori possible que fq soit représenté de façon normalisé par (tracef - fm*t)/2 ...
    # Il existe s entier tel que fq = fm*wk + s.
    # on écrit b = (xf + yf*fq)/fm, xf, yf sont dans ZZ
    
    s1 = (-fm*dk + tracef)/2
    (x,y) = base_fwk(b,dk,f)
    xf = fm*x - y*s1*f
    yf = y*f
    return xf,yf


def eval_fq(P,kq,E):

    #calcul Frobenius(P) sur une courbe E définie sur Fq
    #Le point P peut appartenir à un extension, il faut alors donner E telle que P appartienne à E.
    if P == E(0):
        return P
    else :
        frob = EllipticCurveHom_frobenius(E, kq)
        #Q = [P[0],P[1]]
        #Q[0] = (Q[0]).pth_power(kq)
        #Q[1] = (Q[1]).pth_power(kq)
        #Q = E(Q)
        return frob(P)

In [42]:
def facto_id_friable(L,O,N,factoN):
    #On suppose L de norme N friable déjà factorisé par des premiers splits, selon factoN.
    assert L.norm() == N
    
    D = O.discriminant()
    qL = L.quadratic_form()
    b = qL[1]
    B = O
    factoL = []
    expoL = []
    for facteur in factoN:
        p = facteur[0]
        exp = facteur[1]
        qp = forme_de_norme(p,D)
        bp = qp[1]
        Ip = NumberFieldOrderIdeal(O,qp)
        expoL.append(exp)
        for _ in range(exp):
            if Mod(b, 2*p) == Mod(bp, 2*p):
                B = Ip*B
                conjug = False
            else:
                B = (Ip.conjugate())*B
                conjug = True
        if conjug:
            factoL.append(Ip.conjugate())
        else:
            factoL.append(Ip)
    assert B.is_equivalent(L)
    assert B.norm() == N

    return factoL, expoL

#L_friable = liste_ideq[130][0]
#N = liste_ideq[130][2]
#factoN = liste_ideq[130][3]
#factoL, expoL = facto_id_friable(L_friable,O,N,factoN)

In [74]:
def partie_friable(L,Ne):
    #On suppose Ne être la partie friable de la norme de L, pour une certaine famille de nombre premier.
    
    D = O.discriminant()
    qL = L.quadratic_form()
    
    N = L.norm()
    assert N%Ne == 0
    a = qL[0]
    m = N/a
    assert m.is_square()
    m = isqrt(m)
    
    Ne2 = a.gcd(Ne)
    me = Ne/Ne2
    assert me.is_square()
    me = isqrt(me)
    assert Ne == Ne2*(me^2)
    
    factoL = []
    expoL = []
    if Ne2 > 1:
        facto_Ne2 = Ne2.factor()
        b = qL[1]
        B = O
        for facteur in facto_Ne2:
            p = facteur[0]
            exp = facteur[1]
            qp = forme_de_norme(p,D)
            bp = qp[1]
            Ip = NumberFieldOrderIdeal(O,qp)
            expoL.append(exp)
            for _ in range(exp):
                if Mod(b, 2*p) == Mod(bp, 2*p):
                    B = Ip*B
                    conjug = False
                else:
                    B = (Ip.conjugate())*B
                    conjug = True
            if conjug:
                factoL.append(Ip.conjugate())
            else:
                factoL.append(Ip)

    if me > 1:
        factoL.append(me*O)
        expoL.append(1)

    return factoL, expoL

In [75]:
L = ideal_de_norme(31,O.conductor(),D)
LL = ideal_de_norme(19,O.conductor(),D)
(L).quadratic_form()
partie_friable(19*LL*LL,19*19*19*19)

([Ideal (7/2*rK + 1/2, 19*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I,
  Ideal (19/2*rK + 19/2, 19*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I],
 [2, 1])

In [134]:
#Construction du groupe de torsion E[L], méthode 3 : Recherche d'un polynôme qui décrit le noyau

def coeff_courbe_l_isogene(E,l,h,Psi_l):
    F = E.base_field()
    j = E.j_invariant()
    a = E.a4()
    b = E.a6()
    
    ZXY = Psi_l.parent()
    (X,Y) = ZXY.gens()
    PsiX = (Psi_l.derivative(X))
    PsiX_eval = PsiX(j,h)
    PsiY = (Psi_l.derivative(Y))
    PsiY_eval = PsiY(j,h)
    
    h1 = F(-18/l)*(b/a)*(PsiX_eval/PsiY_eval)*j
    A = F(-1/48)*(h1^2)/((h-1728)*h)
    B = F(-1/864)*(h1^3)/((h-1728)*h^2)
    A = A*(F(l)^4) #Normalisation
    B = B*(F(l)^6)
    return A,B


def CheckElkies(E, ell, kernel_polynomial, lam):   #CF Pegasis Elkies.py
    r"""Given a kernel polynomial, verify it corresponds to the correct eigenvalue

    If the multiplication-by-lambda map has the following standard form in
    rational maps (c.f. Sutherland's lectures)

        [\lambda] = (u(x)/v(x), r(x, y)/s(x))

    then the eigenvalue is correct, if

        \pi(P) = \lambda P

    on all points in the kernel of the isogeny defined by kernel_polynomial.

    Note that r(x, y) = r(x, 1) * y, by the standard form of isogenies. So, if
    P = (x, y), this is equivalent to

        (x^p, y^p) = (u(x)/v(x), r(x, 1)/s(x) * y)

    for points in ker(\varphi)

    Verifying the first component is easy. To verify the second, we note that

            y^p = r(x, 1)/s(x) * y
        <=> y^{p-1} = r(x, 1)/s(x)
        <=> f(x)^{(p-1)/2} = r(x, 1)/s(x)
        <=> f(x)^{(p-1)/2} * s(x) = r(x, 1)

    where f(x) is the defining equation of the curve E: y^2 = f(x).
    """

    p = E.base_field().characteristic()
    E = E.short_weierstrass_model()

    # Defining equation of E: y^2 + g(x)y = f(x)
    f, g = E.hyperelliptic_polynomials()

    # Must be true, because E is in Weierstrass form
    assert g == 0

    # For efficiency: replace lambda with -lambda if -lambda has smaller
    # absolute value
    # If we switch, then we need to multiply the isogeny with -1
    # (which is multiplication by -1 on the y-coordinate)

    if lam > ell - lam:
        lam = ell - lam
        sign = -1
    else:
        sign = 1

    if lam == 1:
        Y = pow(f, (p - 1) / 2, kernel_polynomial)
        return sign * Y == 1

    # Build extension over which the x-coordinates of the kernel are defined
    extension = kernel_polynomial.parent().quotient_ring(kernel_polynomial)

    # Get rational functions of multiplication-by-lambda
    # x = u/v, y = r/x as in the description in the docstring
    x, y = E.multiplication_by_m(lam)[:2]
    u = extension(x.numerator())
    v = extension(x.denominator())
    # Overwrite r(x, y) with r(x, 1)
    r = y.numerator()
    r = extension(r(r.variables()[0], 1).univariate_polynomial())
    s = extension(y.denominator())

    # x^p in the extension
    Xp = extension(pow(kernel_polynomial.parent().gens()[0], p, kernel_polynomial))

    # Verify u(x)/v(x) = x^p
    if u != Xp * v:
        return False

    Y = extension(pow(f, (p - 1) / 2, kernel_polynomial))

    # Verify y^{p-1} = f(x)^{(p-1)/2} = sign * r(x, 1)/s(x)
    return Y * s == sign * r



def polynome_ker(E,l,L,f,fm,dk,tracef,j,Psi_ni):

    assert gcd(fm,l) == 1   #assure qu'il n'existe que deux isogénies rationnelles. 

    ql = L.quadratic_form()
    cible = (-ql[1] + f*rK)/2                    #Deuxieme élément d'une base sous forme standard .
    (c,d) = base_fq1(cible, fm, dk, f, tracef)
    val = Mod(-c,l)*(Mod(d,l)^(-1))   # d = 1 ?
    val = ZZ(val)
    if abs(val) > abs(val - l):   #necessaire ?
        val = val - l
    
    PFq.<z> = PolynomialRing(Fq)
    Psi = Psi_ni(j,z)
    racines = Psi.roots()
    assert len(racines) == 2 or racines[0][1] == 2, f"racines was {racines}"
    h = racines[0][0]

    assert h.parent() == E.base_ring()
    #test_j_invariant = (gcd(Psi,z^q - z)).factor()
    #h = -(test_j_invariant[0][0][0])
    (A,B) = coeff_courbe_l_isogene(E,l,h,Psi_ni)
    E_coef = EllipticCurve(Fq,[A,B])
    Fker = compute_isogeny_bmss(E, E_coef, l) #On suppose p > 4l + 4 
    
    if CheckElkies(E, l, Fker, val):
        return Fker
    else:
        h = racines[1][0]
        (A,B)=coeff_courbe_l_isogene(E,l,h,Psi_ni)
        E_coef = EllipticCurve(Fq,[A,B])
        Fker = compute_isogeny_bmss(E, E_coef, l)
        if CheckElkies(E, l, Fker, val):
            return Fker
        else:
            raise 'Aucun j trouvé'

In [131]:
def phi_from_L_polynome(E,L,l,q,kq,f, fm, dk, tracef ,j ,Psi_ni):

    #résume ce qui précéde pour passer d'un idéal de norme l à une isogénie de degré l.
    print('Calcul du polynome du noyau')
    F_ker = polynome_ker(E,l,L,f,fm,dk,tracef,j,Psi_ni)
    print('Calcul de Vélu')
    phi = E.isogeny(F_ker) #même syntaxe qu'avec un générateur 
    return phi


def composantes_de_phi_polynome(E_eval,q,n,kq,Exposant,Idéaux,f,fm,dk,tracef):

    #Algo 4 de Jao (Algo de Broker) : On déduit de la factorisation de L une suite d'isogénies.
    
    dépard = E_eval 
    j = dépard.j_invariant()
    k = len(Idéaux)
    composantes = []
    PPFq.<X,Y> = PolynomialRing(Fq, order = 'lex')
    for i in [0 .. k-1]:
        pi = Idéaux[i]
        ni = int(pi.norm())
        Psi_ni = classical_modular_polynomial(ni)  #Acces à une database. Par défaut majoré par 100
        print('Composante de degré :')
        print(pi.norm())
        print('Exposant :')
        print(Exposant[i])  
        for i in range(Exposant[i]):
                print('calcul composante numéro', i+1)
                j = dépard.j_invariant()
                phic = phi_from_L_polynome(dépard,pi,ni,q,kq,f,fm,dk,tracef,j,Psi_ni)
                composantes.append(phic)
                dépard = phic.codomain()
                dépard = EllipticCurve(GF(q^n), [dépard.a4(),dépard.a6()]) #On se replace sur le corps de base pour éviter une escalade d'extensions
    return composantes

In [133]:
p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

kq = 1
q = p^kq
Fq = GF(q)

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

dK = K.discriminant()
f = O.conductor()
D = (f^2)*dK

tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

classical_modular_polynomial.set_cache_bound(500) 
composantes_de_phi_polynome(E,p,1,1,expoL,factoL,f,fm,dK,tracef)

Composante de degré :
5
Exposant :
2
calcul composante numéro 1
Calcul du polynome du noyau
Calcul de Vélu
calcul composante numéro 2
Calcul du polynome du noyau
Calcul de Vélu
Composante de degré :
173
Exposant :
1
calcul composante numéro 1
Calcul du polynome du noyau
Calcul de Vélu
Composante de degré :
251
Exposant :
1
calcul composante numéro 1
Calcul du polynome du noyau
Calcul de Vélu


NotImplementedError: modular polynomial is not in database and computing it on the fly is not yet implemented

In [113]:
Psi_3 = classical_modular_polynomial(3)
j = E.j_invariant()
PFq.<z> = PolynomialRing(Fq)
Psi = Psi_3(j,z)
Psi.roots()[1][0]

20993353057469952958741438509475151757292141603436942089426042761402652614770

In [65]:
fm.factor()

3

In [66]:
fm

3